# Convert Question to Solution Approach
This notebook has the code to use an LLM to find the solution for a question and convert that to a solution approach. Then, we can cluster questions based on the solution approach, rather than just similarity of questions.

In [1]:
%pip install openai
%pip install dotenv


[notice] A new release of pip is available: 23.2.1 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 23.2.1 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [7]:
import asyncio
import os
import re
from dotenv import load_dotenv
from openai import OpenAI

In [5]:
load_dotenv()  # read .env file
api_key = os.getenv("OPENAI_API_KEY")

In [6]:
filename = 'output/inv-trig/rep-questions.md'
rep_questions = []
with open(filename, 'r') as f:
    rep_questions = f.readlines()

In [4]:
def extract_number(question):
    m = re.match(r'^\s*(\d+)\.\s+', question)
    if m:
        number = int(m.group(1))
        return number
    return 0

In [8]:
client = OpenAI()
approaches = {}
for question in rep_questions:
    number = extract_number(question)
    print(f"Getting summary for solution of question: {number}")
    response = client.chat.completions.create (
                            model="gpt-5-mini",
                            messages=[
                                {"role": "system", 
                                "content": 
                                        """You are a Math specialist who can provide solution approaches 
                                        to senior level math problems.
                                        You will be provided a Math question encoded using Latex syntax for
                                        formulae and other symbols. Your task is to solve the problem. However, 
                                        instead of providing the solution, you are supposed to summarize the 
                                        approach to solve the problem by listing the micro-skills required to
                                        solve the problem. You must use standardized tags in recording these
                                        micro-skills. The summary must be concise and short,
                                        typically, just using one line. It should describe just enough to catch the 
                                        key concepts, topics and micro-skills that will be involved in solving 
                                        the problem. For example, a concise and short approach could read: 
                                        Analyze expression inside arccos, find its minimum/maximum, monotonicity, 
                                        range mapping. Avoid using specific numbers in the response unless the usage 
                                        of numbers is critical to the solution approach. If symbols are being used, 
                                        please convert to a latex encoding as provided in the question. Do not use 
                                        symbols for greek variables and mathematical symbols. Return the concise 
                                        approach in a single line. Do not break into multiple lines.
                                        """},
                                {"role": "user",
                                "content": [
                                        {"type": "text",
                                        "text": 
                                            f"""
                                            Return the solution approach including micro skills for solving this problem:
                                
                                            {question}
                                            """ 
                                        }
                                ]}
                            ]
                )
    approaches[number] = response.choices[0].message.content 
    print(f"{number}: {response.choices[0].message.content}")
    asyncio.sleep(5)

Getting summary for solution of question: 1
1: [PRINCIPAL-VALUE] Identify principal range of \cos^{-1} as [0,\pi]; [EVALUATE] use standard angles to evaluate \cos^{-1}\!\left(\frac{1}{\sqrt{2}}\right) and \cos^{-1}\!\left(-\frac{\sqrt{3}}{2}\right); [UNIT-CONVERSION] convert both to same units (degrees or radians); [ARITHMETIC] compute their difference; [MODULAR-ADJUST] adjust by 2\pi (or 360°) if needed to match conventional positive angle; [MATCH] select the matching choice.
Getting summary for solution of question: 5


/tmp/ipykernel_5465/3353085558.py:44: RuntimeWarning: coroutine 'sleep' was never awaited
  asyncio.sleep(5)


5: [DomainRange] Identify domains/ranges of \\(\\sin^{-1},\\cos^{-1},\\sec^{-1}\\); [InverseComposition] Use identity \\(f(f^{-1}(x))=x\\) for \\(x\\) in domain of \\(f^{-1}\\); [Transform] Express \\(\\sec^{-1}x=\\cos^{-1}(1/x)\\) to relate to \\(\\cos\\circ\\cos^{-1}\\); [PrincipalValue] Account for principal-value (branch) definitions of inverse trig functions; [Counterexample/Test] Verify each option by plugging representative \\(x\\) values from the specified intervals.
Getting summary for solution of question: 6
6: [PRINCIPAL_RANGE] Identify principal value range of \(\sin^{-1}\): \([-\pi/2,\pi/2]\); [PERIODICITY] reduce \(x\) by integer multiples of \(2\pi\) using \(\sin\) periodicity; [SYMMETRY] apply sine symmetry identities to map angles into \([-\pi/2,\pi/2]\); [MAPPING] find equivalent \(a\in[-\pi/2,\pi/2]\) with \(\sin a=\sin x\) and write \(\sin^{-1}(\sin x)=a\).
Getting summary for solution of question: 8
8: [Trigonometric identities] Reduce angle modulo \pi for \cot\lef